In [1]:
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
GOLD_DB = PROJECT_ROOT / "data" / "gold" / "fraud_dev.duckdb"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)


def q(sql):
    """Run a query on the Gold warehouse (read-only) and return a DataFrame."""
    with duckdb.connect(str(GOLD_DB), read_only=True) as con:
        return con.sql(sql).df()


print("Gold file exists:", GOLD_DB.exists())
q("""
    select count(*)      as n_rows,
           sum(is_fraud) as n_fraud
    from fact_transactions
""")

Gold file exists: True


,n_rows,n_fraud
0,1852394,9651.0


In [2]:
monthly = q("""
    select strftime(trans_ts, '%Y-%m')                       as month,
           count(*)                                          as n_txn,
           cast(sum(is_fraud) as integer)                    as n_fraud,
           round(100 * avg(is_fraud), 3)                     as fraud_pct,
           count(distinct case when is_fraud = 1
                               then card_key end)            as fraud_cards
    from fact_transactions
    group by 1
    order by 1
""")

print("total rows :", monthly["n_txn"].sum())
print("total fraud:", monthly["n_fraud"].sum())
monthly

total rows : 1852394
total fraud: 9651


,month,n_txn,n_fraud,fraud_pct,fraud_cards
0,2019-01,52525,506,0.963,50
1,2019-02,49866,517,1.037,53
2,2019-03,70939,494,0.696,49
3,2019-04,68078,376,0.552,41
4,2019-05,72532,408,0.563,42
5,2019-06,86064,354,0.411,35
6,2019-07,86596,331,0.382,36
7,2019-08,87359,382,0.437,39
8,2019-09,70652,418,0.592,44
9,2019-10,68758,454,0.660,50


In [3]:
from src.models.split import TRAIN, GAP, VALIDATION, TEST, assign_split

tx = q("select trans_ts, is_fraud, card_key from fact_transactions")
tx["split"] = assign_split(tx["trans_ts"])

fraud_rows = tx[tx["is_fraud"] == 1]
split_summary = pd.DataFrame({
    "n_txn": tx.groupby("split").size(),
    "n_fraud": tx.groupby("split")["is_fraud"].sum(),
    "fraud_cards": fraud_rows.groupby("split")["card_key"].nunique(),
}).reindex([TRAIN, GAP, VALIDATION, TEST])
split_summary["fraud_pct"] = (100 * split_summary["n_fraud"] / split_summary["n_txn"]).round(3)

print("total rows :", split_summary["n_txn"].sum())
print("total fraud:", split_summary["n_fraud"].sum())
split_summary

total rows : 1852394
total fraud: 9651


,n_txn,n_fraud,fraud_cards,fraud_pct
split,,,,
train,1306680,7528,762,0.576
gap,36006,208,23,0.578
validation,228187,979,102,0.429
test,281521,936,94,0.332


In [4]:
# Which splits does each fraud card appear in?
card_splits = (
    fraud_rows.groupby("card_key")["split"]
    .agg(lambda s: " + ".join(sorted(set(s))))
)
print("fraud cards in total:", card_splits.size)
card_splits.value_counts()

fraud cards in total: 976


split
train               762
validation           98
test                 93
gap                  18
gap + validation      4
gap + test            1
Name: count, dtype: int64